# Mod8L5 PySpark vs. pandas (Local Parquet) — Do in Google Colab
**Goal:** See the syntax and workflow differences between **pandas** and **PySpark** using a **local Parquet** file (no cloud).  
**Format:** INSTRUCTOR → YOU DO → REFLECTION

## INSTRUCTOR — Follow Along Together (20 mins.)

### Why Parquet (local-first)
- **Columnar** → fast column scans & smaller I/O.
- **Compressed & splittable** → efficient storage & parallel reads.
- **Self-describing schema** → types are embedded in the file.
- **Predicate pushdown** (Spark) → skip non-matching row groups.

**Docs:**  
- Spark SQL/DataFrames: https://spark.apache.org/docs/latest/sql-programming-guide.html  
- Spark Reader/Writer Parquet: https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameReader.parquet.html  
- pandas `read_parquet`/`to_parquet`: https://pandas.pydata.org/docs/reference/api/pandas.read_parquet.html

### Step 0 — (Optional) Install libraries (only if needed)


In [ ]:
# Do a Pip Install in the Terminal for these pacakages

#pip install pyspark==3.5.1
#pip install pyarrow==15.0.2


### Step 1 — Create a tiny local Parquet (once) and keep it on disk
We’ll generate a small sample dataset, save it to **`./customers.parquet/`** (a folder of Parquet part files).

In [2]:
# RUN THIS CELL WITHOUT CHANGES
import pandas as pd
import numpy as np

rng = np.random.default_rng(42)
n = 20
pdf = pd.DataFrame({
    "customer_id": np.arange(1, n+1),
    "age": rng.integers(18, 70, size=n),
    "country": rng.choice(["US", "UK", "FR", "DE"], size=n),
    "spend_usd": rng.normal(100, 30, size=n).round(2)
})

# Save locally as Parquet (folder)
pdf.to_parquet("customers.parquet", index=False)
print("Wrote local parquet folder: ./customers.parquet/")


Wrote local parquet folder: ./customers.parquet/


In [3]:
pdf2 = pd.read_parquet("customers.parquet")
display(pdf2.head())
display(pdf2.dtypes)

pdf_country = (
    pdf2.groupby("country", as_index=False)
        .agg(n_rows=("customer_id","count"),
             avg_spend=("spend_usd","mean"))
        .sort_values("avg_spend", ascending=False)
)
display(pdf_country)


,customer_id,age,country,spend_usd
0,1,22,FR,94.45
1,2,58,UK,79.57
2,3,52,US,136.68
3,4,40,DE,95.36
4,5,40,DE,87.15


,0
customer_id,int64
age,int64
country,object
spend_usd,float64


,country,n_rows,avg_spend
3,US,4,110.747500
2,UK,5,109.596000
1,FR,5,98.276000
0,DE,6,97.221667


### Step 3 — Start SparkSession (local) and read the same local Parquet

**Instructor:** You can hover over methods in Google Colab to see the documentation.  Note to students what is similar between this syntax and pandas syntax


In [4]:
# RUN THIS CELL WITHOUT CHANGES
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("local_parquet_demo").getOrCreate()
df = spark.read.parquet("customers.parquet")
df.printSchema()
df.show(5, truncate=False)




root
 |-- customer_id: long (nullable = true)
 |-- age: long (nullable = true)
 |-- country: string (nullable = true)
 |-- spend_usd: double (nullable = true)

+-----------+---+-------+---------+
|customer_id|age|country|spend_usd|
+-----------+---+-------+---------+
|1          |22 |FR     |94.45    |
|2          |58 |UK     |79.57    |
|3          |52 |US     |136.68   |
|4          |40 |DE     |95.36    |
|5          |40 |DE     |87.15    |
+-----------+---+-------+---------+
only showing top 5 rows


### Step 4 — Spark: do the same (select, filter, groupBy)


In [5]:
# Select a few columns
df_sel = df.select("customer_id", "country", "spend_usd")

# Filter
df_us = df_sel.filter(F.col("country")== F.lit("US"))
print("US rows:", df_us.count())
df_us.show(truncate=False)

# Group & aggregate (Spark)
df_country = (
    df.groupBy("country")
    .agg(F.count("*").alias("n_rows"),
         F.avg("spend_usd").alias("avg_spend"))
    .orderBy(F.col("avg_spend").desc())
)
df_country.show(truncate=False)


US rows: 4
+-----------+-------+---------+
|customer_id|country|spend_usd|
+-----------+-------+---------+
|3          |US     |136.68   |
|12         |US     |87.81    |
|13         |US     |84.63    |
|16         |US     |133.87   |
+-----------+-------+---------+

+-------+------+-----------------+
|country|n_rows|avg_spend        |
+-------+------+-----------------+
|US     |4     |110.7475         |
|UK     |5     |109.596          |
|FR     |5     |98.276           |
|DE     |6     |97.22166666666665|
+-------+------+-----------------+



### Step 5 — Write a filtered subset to Parquet & round-trip


In [6]:
out_path = "customers_high_spend.parquet"
(
    df.filter(F.col("spend_usd") > 100)
      .write.mode("overwrite")
      .parquet(out_path)
)
rt = spark.read.parquet(out_path)
rt.show(truncate=False)


+-----------+---+-------+---------+
|customer_id|age|country|spend_usd|
+-----------+---+-------+---------+
|3          |52 |US     |136.68   |
|7          |22 |UK     |115.97   |
|8          |54 |DE     |110.96   |
|9          |28 |FR     |112.38   |
|10         |22 |UK     |112.92   |
|11         |45 |UK     |164.25   |
|15         |55 |DE     |118.48   |
|16         |58 |US     |133.87   |
|20         |41 |FR     |119.52   |
+-----------+---+-------+---------+



## YOU DO (30 mins)

Use the docs to complete the tasks in Spark **and** (where it helps you think) pandas.

**Helpful Docs:**  
- Spark `functions`: https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html  
- Spark `DataFrame` API: https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/index.html  
- pandas groupby: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html

### Task A — Schema, sample, and simple transforms (Spark)
1. Print the schema and count rows.  
2. Add two columns:
   - `spend_eur = spend_usd * 0.92`
   - `is_senior = age >= 40` (boolean)
3. Show the top 8 rows sorted by `spend_usd` descending.

### Task B — Aggregations (Spark)
4. By `country`, compute:
   - `n_rows = count(*)`
   - `avg_age = avg(age)`
   - `p90_spend = percentile_approx(spend_usd, 0.90)` (use `F.expr(...)` or SQL expr)
   Sort by `p90_spend` descending, show all rows.

### Task C — (Compare) Do the same aggregation in pandas
5. Load the same `customers.parquet` into pandas and compute:
   - `n_rows`, `avg_age`, and the 90th percentile of `spend_usd` (use `quantile(0.90)`).
   - Sort to match Spark output.

> Tip: Notice Spark’s **lazy** execution vs pandas’ **eager** execution.


In [7]:
# Task A
print(df.printSchema())
print(f"Row Count: {df.count()}")

root
 |-- customer_id: long (nullable = true)
 |-- age: long (nullable = true)
 |-- country: string (nullable = true)
 |-- spend_usd: double (nullable = true)

None
Row Count: 20


In [10]:
df_new = df.withColumn('spend_eur', F.col('spend_usd') * 0.92)
df_new = df.withColumn('is_senior', F.col('age')>= 40)
df_new.show(5, truncate=True)

+-----------+---+-------+---------+---------+
|customer_id|age|country|spend_usd|is_senior|
+-----------+---+-------+---------+---------+
|          1| 22|     FR|    94.45|    false|
|          2| 58|     UK|    79.57|     true|
|          3| 52|     US|   136.68|     true|
|          4| 40|     DE|    95.36|     true|
|          5| 40|     DE|    87.15|     true|
+-----------+---+-------+---------+---------+
only showing top 5 rows


In [15]:
df_country_rows = (
    df_new.groupBy("country")
    .agg(F.count("*").alias("n_rows"),
         F.avg("age").alias("avg_age"),
         F.expr("percentile_approx(spend_usd, 0.90)").alias("p90_spend"))
    .orderBy(F.col("p90_spend").desc())
)
df_country_rows.show(truncate=True)

+-------+------+------------------+---------+
|country|n_rows|           avg_age|p90_spend|
+-------+------+------------------+---------+
|     UK|     5|              41.6|   164.25|
|     US|     4|              58.5|   136.68|
|     FR|     5|              42.0|   119.52|
|     DE|     6|42.833333333333336|   118.48|
+-------+------+------------------+---------+



In [17]:
pdf_stats = (pdf
             .groupby("country", as_index=False)
             .agg(n_rows=("customer_id","count"),
                  avg_age=("age","mean"),
                  p90_spend=("spend_usd", lambda s: s.quantile(0.90)))
             .sort_values("p90_spend", ascending=False)
             )
display(pdf_stats)

,country,n_rows,avg_age,p90_spend
2,UK,5,41.600000,144.938
3,US,4,58.500000,135.837
1,FR,5,42.000000,116.664
0,DE,6,42.833333,114.720


## REFLECTION — Short Answers


1) **Why Parquet locally?**
- If you used CSV yesterday and Parquet today for the same data, what changes in performance and why?

2) **Scaling Up**
- If this dataset were 100 GB on your laptop, which tool would likely still run and why?  
- What would you change first to handle that scale?

